In [1]:
import sys
sys.path.append("..")

# Need Compile Session
search for MODIFY to FILL IN

In [ ]:
threshold = 2  # Choose a threshold for minimum samples has the same source object and the same target object
task_type = 'what relation'  # Choose a task plan from ['what object', 'what attribute', 'what relation']

In [2]:
from tma.imageqa.scene_graph import *
from tma.base import JointTaskGenerator
from tma.imageqa.metadata import SceneGraphMetaData


path = '../TaskMeAnything-v1-source/vg'  # the path to scene_graph folder
metadata = SceneGraphMetaData('../annotations', scene_graph_folder=path)

generators = {
    'what object'              : WhatObjectSceneGraphTaskGenerator,
    'what attribute'             : WhatAttributeSceneGraphTaskGenerator,
    'what relation'          : WhatRelationSceneGraphTaskGenerator,
}
generator = JointTaskGenerator(metadata, generators)

In [4]:
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

save_path = '../demo/cache/sg.parquet' 
df = pd.read_parquet(save_path)
display(df.tail())


,task type,object,subgraph,scene graph id,answers,attribute type,attribute value,relation,source object,target object,source subgraph,target subgraph
798182,what relation,None,None,2379672,[to the right of],None,None,to the right of,curtain,hair,"{""attributes"": [""floral"", ""flowered"", ""open""],...","{""attributes"": [""dark"", ""long""], ""adjacent_obj..."
798183,what relation,None,None,2379672,[to the right of],None,None,to the right of,curtain,child,"{""attributes"": [""flowered""], ""adjacent_objects...","{""attributes"": [""small""], ""adjacent_objects"": ..."
798184,what relation,None,None,2379672,[to the right of],None,None,to the right of,curtain,cat,"{""attributes"": [""flowered"", ""open""], ""adjacent...","{""attributes"": [""black"", ""sitting""], ""adjacent..."
798185,what relation,None,None,2379672,[to the right of],None,None,to the right of,curtain,toilet,"{""attributes"": [""flowered""], ""adjacent_objects...","{""attributes"": [""reflective""], ""adjacent_objec..."
798186,what relation,None,None,2379672,[to the left of],None,None,to the left of,toilet,curtain,"{""attributes"": [""reflective""], ""adjacent_objec...","{""attributes"": [""floral"", ""flowered""], ""adjace..."


In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from tma.task_store import get_pd_schema
from tqdm import tqdm
from pprint import pprint # use pprint to print the task
from IPython.display import display

save_path = '../demo/cache/sg.parquet'
filter = [('task type', '==', task_type)]
df = pq.read_table(save_path, filters=filter).to_pandas().astype(get_pd_schema(generator.schema))

print('num of unique target objects:')
print(len(set(df['target object'])))
print('num of unique source objects:')
print(len(set(df['source object'])))

grouped_counts = df.groupby(['source object', 'target object']).size()

average_count = grouped_counts.mean()

median_count = grouped_counts.median()

# print("Counts of samples per group:")
# print(grouped_counts)

print("Average number of samples per group:")
print(average_count)

print("\Median number of samples per group:")
print(median_count)

sorted_counts = grouped_counts.sort_values(ascending=False)
print(f"Counts of samples per group > {threshold}, ordered by most frequent:")
print(len(sorted_counts[sorted_counts > threshold]))

In [ ]:
filtered_df = df.groupby(['source object', 'target object'])

grouped_counts = df.groupby(['source object', 'target object']).size()
sorted_counts = grouped_counts.sort_values(ascending=False)
thresholded_counts = sorted_counts[sorted_counts >= threshold]

display(thresholded_counts)

# Generate all in-context tasks
output: tasks -> list[list[dict, dict]]

In [19]:
import pickle 
tasks = []

for source, target in thresholded_counts.index:
    filtered_samples = df[(df['source object'] == source) & (df['target object'] == target)]

    for idx in filtered_samples.index:
        row_data = df.iloc[idx].dropna().to_dict()
        task = generator.generate(row_data, return_data=True)
        remaining_samples = filtered_samples.drop(idx)

        if not remaining_samples.empty:
            random_idx = remaining_samples.sample(n=1).index[0] 
            sampled_row_data = df.iloc[random_idx].dropna().to_dict()
            sampled_task = generator.generate(sampled_row_data, return_data=True)

            print(f"Original task: {task}\n")
            print(f"Randomly sampled task: {sampled_task}\n")
            tasks.append([sampled_task, task])
        else:
            # Raise an exception if only one row is available
            raise Exception("Insufficient samples. Need to set threshold >= 2 to ensure multiple samples.")
    #     break  # MODIFY [for visual purpose only]
    # break   # MODIFY [for visual purpose only]

with open(f'scene_graph_{task_type}_tasks.pkl', 'wb') as f:
    pickle.dump(tasks, f)

Original task: {'question': 'What is the relation from the green object, which is to the left of the smiling and young girl, to the pink object, which is to the right of the happy and smiling woman?', 'answer': 'to the left of', 'options': ['to the left of', 'parked alongside', 'pulled by', 'parked near'], 'task_plan': 'task type: what relation\nrelation: to the left of\nsource object: shirt\ntarget object: shirt', 'scene_graph_id': '2394852', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x333 at 0x7FDC24827970>}

Randomly sampled task: {'question': 'What is the relation from the brown and floral object, which is to the left of the sitting girl, to the gray object, which the happy and young woman is to the left of?', 'answer': 'to the left of', 'options': ['to the left of', 'grazing in', 'stacked on', 'walking towards'], 'task_plan': 'task type: what relation\nrelation: to the left of\nsource object: shirt\ntarget object: shirt', 'scene_graph_id': '2403230', 'image

# CLIP Answer Evaluator

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
!pip install ftfy regex tqdm

In [2]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

In [ ]:
import torch.nn.functional as F
def check_ans_to_options(ans, gt_ans, options, model):
    if ans in options: 
        return 1 if options.index(ans) == options.index(gt_ans) else 0
    # use clip to evaluate similarity between ans and options
    options_token = clip.tokenize(options).to(device)
    ans_token = clip.tokenize(ans).to(device)
    with torch.no_grad():
        options_embed = model.encode_text(options_token)
        ans_embed = model.encode_text(ans_token)
    options_embed /= options_embed.norm(dim=-1, keepdim=True)
    ans_embed /= ans_embed.norm(dim=-1, keepdim=True)
    similarity = (ans_embed @ options_embed.T).softmax(dim=-1)
    ans_idx = torch.argmax(similarity)
    gt_idx = options.index(gt_ans)
    return 1 if ans_idx == gt_idx else 0

# example usage
# check_ans_to_options('hello', 'hello', ['hello', 'world', 'goodbye', 'hi'], clip_model)

# Inference

tasks -> list[list[dict(), dict()]], where first dict is in-context example, second dict is the query example

In [ ]:
# example usage
total_tasks = len(tasks)
correct_tasks = 0
for t1, t2 in tasks:
    in_context_q = t1['question']
    in_context_a = t1['answer']
    in_context_o = t1['options']
    in_context_img = t1['image']

    q = t2['question']
    gt = t2['answer']
    o = t2['options']
    img = t2['image']

    prompt = ... # MODIFY
    model = ...  # MODIFY
    ans = model(prompt)
    correct_tasks += check_ans_to_options(ans, gt, o, clip_model)
    
accuracy = correct_tasks / total_tasks
print(f'Accuracy for {task_type} task is: {accuracy}')

